# 03 · Human coding: reliability, validity of the index, parts and whole, expert panel

**What this notebook does.** Four coders coded the same 30 videos (blind sample drawn in notebook 01). This notebook computes
Krippendorff's alpha for every code (four coders), the validity of the index against human template-conformity (V1) in three versions
(all codes; only codes reliable across four coders; only the five codes all experts rated relevant), tests that fit ceiling-heavy, tied and binary
codes (Kendall tau-b, Cliff's delta, element by element), Adorno's "interchangeable parts" checks (segments, choruses, K-pop vs Korean indie,
indie videos with performers), and the content-validity indices of the four-expert panel (I-CVI, modified kappa, S-CVI/Ave).

**Run order.** Notebook 01 must have produced `per_video/` and `tables/indie_chorus_check.xlsx`. Then run top to bottom.
In this notebook the coding table is called `CODE` and the video vectors `V`.

**Thesis.** 3-6-3, 3-6-4; results 4-3-2, 4-3-3, 4-3-4, 4-3-6, 4-8-5 (Tables 3-5b, 4-4, 4-5).

---

## ۰۳ · کدگذاری انسانی: پایایی، روایی شاخص، جزء و کل، پنل خبرگان
پایایی چهار کدگذار (آلفای کریپندورف)، روایی شاخص (V1) در سه نسخه، آزمون‌های متناسب با اثر سقف، بررسی «جزء جایگزین‌پذیر» در کی‌پاپ و ایندی (و ایندی با اجراکننده)، و شاخص اعتبار محتوایی پنل چهار خبره. جدول کدگذاری در این نوتبوک `CODE` نام دارد.

### 1. Install libraries

**فارسی:** نصب کتابخانه‌ها.

*Thesis: appendix C*

In [ ]:
# 1. Install libraries
# Thesis: appendix C
%pip install -q krippendorff numpy pandas scipy openpyxl xlsxwriter

### 2. PATHS (the only cell to edit)

**فارسی:** مسیرها: تنها خانه‌ای که باید ویرایش شود.

*Thesis: 3-6, 3-10*

In [ ]:
# 2. PATHS
# Thesis: 3-6, 3-10
from pathlib import Path
BASE       = Path.home() / 'KPOP'                         # same layout as notebook 01
METADATA   = BASE / 'MV_metadata_105_verified_v2.xlsx'   # corpus metadata (sheet 'Metadata_105')
OUT        = BASE / 'analysis_v2_0'                       # output folder of notebook 01
EXPERT_DIR = BASE / 'expert_review'                       # expert_review_1.xlsx ... expert_review_4.xlsx (returned panel sheets)
# Inputs expected in OUT/tables (not distributed): coding_v5_coder1.xlsx ... coding_v5_coder4.xlsx (sheets 'Video_level' and 'videos'),
# indie_chorus_check.xlsx (chorus onsets by ear, from notebook 01) and indie_composition_filled.xlsx (performer / animation per indie video).
print('tables folder:', (OUT / 'tables').exists(), '| expert folder:', EXPERT_DIR.exists())

### 3. Imports and helper

**فارسی:** کتابخانه‌ها و تابع نرمال‌سازی (کپی از نوتبوک ۰۱).

*Thesis: —*

In [ ]:
# 3. Imports and the l2n helper (copied from notebook 01)
# Thesis: —
import numpy as np, pandas as pd
def l2n(x, axis=-1):
    return x / np.clip(np.linalg.norm(x, axis=axis, keepdims=True), 1e-12, None)

### 4. Fast load of vectors and index

**فارسی:** بارگذاری سریع بردارهای کی‌پاپ و ایندی و بازسازی شاخص، دقیقاً مانند نوتبوک ۰۱.

*Thesis: 3-3-2*

In [ ]:
# 4. Fast load: saved K-pop and indie embeddings, index rebuilt exactly as in notebook 01 (no figures, no bootstraps, no MIR)
# Thesis: 3-3-2
from scipy.stats import spearmanr, mannwhitneyu
import unicodedata, re, json

def load_corpus(corpus):
    rows = {}
    for d in sorted((OUT / 'per_video' / corpus).iterdir()):
        if not ((d / 'info.json').exists() and (d / 'embeddings.npz').exists()): continue
        info = json.loads((d / 'info.json').read_text(encoding='utf-8')); e = np.load(d / 'embeddings.npz', allow_pickle=True)
        W = l2n(e['clip_embeddings'].astype(np.float64))
        rows[info['file_name']] = dict(dir=d, W=W, starts=e['clip_starts'], ends=e['clip_ends'], dur=float(e['duration_sec']), v=l2n(W.mean(0)), u=W.mean(0))
    return rows

K = load_corpus('kpop'); Ii = load_corpus('kindie')
meta = pd.read_excel(METADATA, sheet_name='Metadata_105')
def _key(x): return re.sub(r'[^0-9a-z가-힣]+', '', unicodedata.normalize('NFC', str(x)).lower())
_kmap = {_key(k): k for k in K}
meta['file_name'] = [_kmap.get(_key(x), x) for x in meta['file_name']]
meta = meta[meta['file_name'].isin(K)].reset_index(drop=True)
names = list(meta['file_name']); fem = (meta['gender'].str.lower() == 'female').to_numpy()
V = np.stack([K[x]['v'] for x in names]); S = V @ V.T; np.fill_diagonal(S, np.nan)
zs = lambda x: (x - np.nanmean(x)) / np.nanstd(x, ddof=1)
def balanced(Smat, f):
    return 0.5 * np.nanmean(np.where(f[None, :], Smat, np.nan), 1) + 0.5 * np.nanmean(np.where(~f[None, :], Smat, np.nan), 1)
T = meta[['file_name', 'artist', 'song', 'gender', 'release_year']].copy()
T['index_balanced'] = balanced(S, fem); T['index_balanced_z'] = zs(T['index_balanced'])
print('K-pop videos:', len(names), '| indie videos:', len(Ii))

### 5. Four-coder reliability and V1 in three versions

**فارسی:** پایایی هر کد میان چهار کدگذار (۳۰ ویدئو) و روایی شاخص V1 در سه نسخه.

*Thesis: 3-6-4; 4-3-2, 4-3-3; Tables 4-4, 4-5*

In [ ]:
# 5. Four coders, all 30 videos, final coding manual: Krippendorff alpha of every code + V1 (all codes / reliable codes / expert-endorsed codes)
#   Afterwards CODE holds the four-coder means per video. -> tables/coding_v5_results.xlsx
# Thesis: 3-6-4; 4-3-2, 4-3-3; Tables 4-4, 4-5
import krippendorff, unicodedata, re
from scipy.stats import spearmanr
EL = ['el_group_dance_chorus', 'el_chorus_same_choreo', 'el_signature_move', 'el_dance_break', 'el_formation_pose',
      'el_solo_closeups', 'el_many_outfits', 'el_narrative_insert', 'el_title_card']
SH = ['share_sync_dance', 'share_camera_address', 'share_built_set']
LEVEL = {**{c: 'nominal' for c in EL}, **{c: 'ordinal' for c in SH}, 'typicality_1to5': 'ordinal', 'performers_gender': 'nominal', 'sexualized_framing': 'ordinal'}
EXPERT5 = ['el_group_dance_chorus', 'el_signature_move', 'el_solo_closeups', 'el_title_card']   # + share_sync_dance: the 5 codes all four experts rated relevant
C5 = {}
for k in (1, 2, 3, 4):
    d = pd.read_excel(OUT / 'tables' / f'coding_v5_coder{k}.xlsx', sheet_name='Video_level')
    d = d[d['code_id'].astype(str).str.fullmatch(r'V\d\d')].copy(); d[list(LEVEL)] = d[list(LEVEL)].apply(pd.to_numeric, errors='coerce')
    C5[k] = d.set_index('code_id'); print(f'coder {k}: videos coded =', int(d[EL].notna().any(axis=1).sum()))
ids = sorted(set().union(*[c.index for c in C5.values()]))
def ka(c):
    M = np.array([C5[k][c].reindex(ids).to_numpy(float) for k in C5])
    if np.isfinite(M).sum() < 4 or np.nanstd(M) == 0: return np.nan
    return krippendorff.alpha(reliability_data=M, level_of_measurement=LEVEL[c])
def agree(c):                                          # share of videos where all coders gave the same value (used when alpha is undefined)
    M = pd.DataFrame({k: C5[k][c].reindex(ids) for k in C5}).dropna(); return float((M.nunique(axis=1) == 1).mean()) if len(M) else np.nan
REL5 = pd.DataFrame([dict(code=c, level=LEVEL[c], alpha_four_coders=ka(c), all_coders_agree=agree(c),
                          prevalence_or_mean=float(pd.concat([C5[k][c] for k in C5]).mean())) for c in LEVEL])
REL5['ok_.67'] = REL5.alpha_four_coders >= .67; display(REL5.round(3))

# code_id -> file_name from the 'videos' sheet of the coding files, matched robustly to the analysed file names
_nk = lambda x: re.sub(r'[^0-9a-z\uac00-\ud7a3]+', '', unicodedata.normalize('NFC', str(x)).lower())
_tmap = {_nk(x): x for x in T.file_name}
blind = pd.read_excel(OUT / 'tables' / 'coding_v5_coder1.xlsx', sheet_name='videos')[['code_id', 'file_name']]
blind['file_name'] = [_tmap.get(_nk(x), x) for x in blind.file_name]
print('coded videos matched to the index:', int(blind.file_name.isin(T.file_name).sum()), 'of', len(blind))
allc = pd.concat([C5[k].reset_index() for k in C5])
CODE = allc.groupby('code_id')[EL + SH + ['typicality_1to5', 'sexualized_framing']].mean().reset_index()
CODE = CODE.merge(blind, on='code_id').merge(T[['file_name', 'gender', 'index_balanced_z']], on='file_name')
fem_v = (CODE['gender'].str.lower() == 'female').to_numpy()
def v1_of(els, shs):
    X = l2n(np.c_[CODE[els].to_numpy(float), CODE[shs].to_numpy(float) / 3]); Sp = X @ X.T; np.fill_diagonal(Sp, np.nan)
    tc = zs(balanced(Sp, fem_v)); r, p = spearmanr(tc, CODE['index_balanced_z'])
    bt = [spearmanr(*pd.DataFrame({'a': tc, 'b': CODE['index_balanced_z']}).sample(len(CODE), replace=True, random_state=b).T.to_numpy())[0] for b in range(2000)]
    return dict(rho=r, ci_low=np.nanpercentile(bt, 2.5), ci_high=np.nanpercentile(bt, 97.5), p=p, n=len(CODE), n_codes=len(els) + len(shs))
rel_ok = REL5.set_index('code').alpha_four_coders.ge(.67)
V1_5 = pd.DataFrame([dict(version='all 9 elements + 3 shares (four-coder means)', **v1_of(EL, SH)),
                     dict(version='only codes reliable across four coders (alpha >= .67)', **v1_of([c for c in EL if rel_ok.get(c, False)], [c for c in SH if rel_ok.get(c, False)])),
                     dict(version='only the 5 codes all four experts rated relevant (I-CVI = 1.00)', **v1_of(EXPERT5, ['share_sync_dance']))])
display(V1_5.round(3))
with pd.ExcelWriter(OUT / 'tables' / 'coding_v5_results.xlsx', engine='xlsxwriter') as xw:
    REL5.to_excel(xw, sheet_name='reliability', index=False); V1_5.to_excel(xw, sheet_name='V1', index=False); CODE.to_excel(xw, sheet_name='video_means', index=False)
print('saved:', OUT / 'tables' / 'coding_v5_results.xlsx', '| CODE holds the four-coder means per video')

### 6. Coding vs index with ceiling-appropriate tests; parts and low end

**فارسی:** کدگذاری انسانی در برابر شاخص با آزمون‌های متناسب با سقف و گره‌ها، عنصر به عنصر، جزءهای جایگزین‌پذیر و انتهای پایین.

*Thesis: 3-6-4; 4-3-3, 4-3-4, 4-3-6*

In [ ]:
# 6. Human coding vs index with tests that fit the data (ceiling, ties, binary elements)
#   + Adorno's interchangeable parts (8-s segments) and the low-end check -> tables/v1_parts.xlsx
# Thesis: 3-6-4; 4-3-3, 4-3-4, 4-3-6
from scipy.stats import spearmanr, kendalltau, mannwhitneyu
OUTX = {}; rng = np.random.default_rng(24)
assert isinstance(CODE, pd.DataFrame) and set(EL) <= set(CODE.columns), 'run the four-coder step right before this cell'
CODE = CODE.copy()
VEC = np.stack([K[x]['v'] for x in names]); VEC = VEC / np.linalg.norm(VEC, axis=1, keepdims=True)
fem_c = (CODE['gender'].str.lower() == 'female').to_numpy(); ix = CODE['index_balanced_z'].to_numpy(float)
E = CODE[EL].apply(pd.to_numeric, errors='coerce').to_numpy(float)

def cliff(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); return (np.sign(a[:, None] - b[None, :])).mean()
def boot_ci(fn, n=2000):
    v = [];
    for _ in range(n):
        s = rng.integers(0, len(CODE), len(CODE)); v.append(fn(s))
    return np.nanpercentile(v, [2.5, 97.5])

# (A) Ceiling diagnostics
prev = pd.Series(np.nanmean(E, 0), index=EL)
OUTX['A_element_prevalence'] = prev.rename('prevalence').reset_index().rename(columns={'index': 'element'})
skel = np.nansum(E, 1)                                   # skeleton count: how many template elements a video has
rows = [dict(test='elements with prevalence >= 0.9 (ceiling)', value=int((prev >= .9).sum()), note=f'of {len(EL)}'),
        dict(test='share of videos tied at the most common skeleton count', value=pd.Series(skel).value_counts(normalize=True).iloc[0])]

# (B) Rank association that handles ties: Kendall tau-b, skeleton count vs index
t, p = kendalltau(skel, ix); lo, hi = boot_ci(lambda s: kendalltau(skel[s], ix[s])[0])
rows.append(dict(test='Kendall tau-b, skeleton count vs index (all)', value=t, p=p, ci_low=lo, ci_high=hi, n=len(ix)))
for g, f_ in [('female', fem_c), ('male', ~fem_c)]:
    t, p = kendalltau(skel[f_], ix[f_]); rows.append(dict(test=f'Kendall tau-b, within {g}', value=t, p=p, n=int(f_.sum())))
r, p = spearmanr(skel, ix); rows.append(dict(test='Spearman, skeleton count vs index (for comparison)', value=r, p=p, n=len(ix)))
for c in ['typicality_1to5'] + SH:                       # holistic rating and the three shares (ordinal, many ties)
    if c in CODE:
        x = pd.to_numeric(CODE[c], errors='coerce').to_numpy(float); ok = ~np.isnan(x)
        t, p = kendalltau(x[ok], ix[ok]); rows.append(dict(test=f'Kendall tau-b, {c} vs index', value=t, p=p, n=int(ok.sum())))

# (C) Ends of the scale: top third vs bottom third of the index, cut WITHIN each gender
top = np.zeros(len(ix), bool); bot = np.zeros(len(ix), bool)
for f_ in (fem_c, ~fem_c):
    q1, q2 = np.quantile(ix[f_], [1/3, 2/3]); top |= f_ & (ix >= q2); bot |= f_ & (ix <= q1)
d = cliff(skel[top], skel[bot]); p = mannwhitneyu(skel[top], skel[bot], alternative='two-sided')[1]
bd = []
for _ in range(2000):
    a = rng.choice(skel[top], top.sum()); b = rng.choice(skel[bot], bot.sum()); bd.append(cliff(a, b))
rows.append(dict(test="Cliff's delta, skeleton count, top vs bottom third (within gender)", value=d, p=p,
                 ci_low=np.percentile(bd, 2.5), ci_high=np.percentile(bd, 97.5), n=f'{top.sum()} vs {bot.sum()}',
                 mean_top=skel[top].mean(), mean_bottom=skel[bot].mean()))

# (D) Element by element: do videos WITH the element have a higher index? (only elements that vary)
er = []
for j, c in enumerate(EL):
    has = E[:, j] >= .5; pr = has.mean()
    if pr >= .9 or pr <= .1: er.append(dict(element=c, prevalence=pr, note='skipped: almost constant')); continue
    er.append(dict(element=c, prevalence=pr, n_with=int(has.sum()), cliff_delta_index=cliff(ix[has], ix[~has]),
                   p=mannwhitneyu(ix[has], ix[~has], alternative='two-sided')[1]))
ER = pd.DataFrame(er)
if 'p' in ER:
    ok = ER.p.notna(); o = ER.loc[ok, 'p'].sort_values(); m = len(o)
    holm = np.maximum.accumulate([min(1, (m - i) * v) for i, v in enumerate(o.values)])
    ER.loc[o.index, 'p_holm'] = holm
OUTX['D_element_by_element'] = ER
OUTX['BC_tests'] = pd.DataFrame(rows)

# (E) Adorno's parts: is an 8-second segment closer to segments of OTHER acts' videos than to the rest of its own video?
art = T.set_index('file_name').artist.reindex(names).to_numpy(); idx = zs(balanced(S, fem))
W_ = [np.asarray(K[x]['W'], float) for x in names]; W_ = [w / np.linalg.norm(w, axis=1, keepdims=True) for w in W_]
own = np.concatenate([[i] * len(w) for i, w in enumerate(W_)]); pos = np.concatenate([np.arange(len(w)) for w in W_])
A_ = np.vstack(W_); SIM = A_ @ A_.T
SIM[(own[:, None] == own[None, :]) & (np.abs(pos[:, None] - pos[None, :]) <= 2)] = -np.inf   # drop overlapping/adjacent segments
nn = SIM.argmax(1); swap = pd.Series(art[own[nn]] != art[own]).groupby(own).mean().to_numpy()
pr_ = []
r, p = spearmanr(idx, swap); pr_.append(dict(test="rho(index, share of segments whose nearest segment is in another act's video)", value=r, p=p, n=len(idx)))
for g, f_ in [('female', fem), ('male', ~fem)]:
    r, p = spearmanr(idx[f_], swap[f_]); pr_.append(dict(test=f'same, within {g}', value=r, p=p, n=int(f_.sum())))

# (F) Low end: do the 15 least standardized videos share a second template among themselves?
def coh(i_): s_ = VEC[i_] @ VEC[i_].T; return s_[np.triu_indices(len(i_), 1)].mean()
o_ = np.argsort(idx); rand = np.array([coh(rng.choice(len(names), 15, replace=False)) for _ in range(5000)])
for lab, i_ in [('lowest 15', o_[:15]), ('highest 15', o_[-15:])]:
    c = coh(i_); pr_.append(dict(test=f'coherence among the {lab} videos', value=c, random_median=np.median(rand),
                                 share_random_at_least_as_coherent=np.mean(rand >= c)))
OUTX['EF_parts_low_end'] = pd.DataFrame(pr_)
OUTX['parts_per_video'] = pd.DataFrame(dict(file_name=names, index_z=idx, share_parts_like_other_acts=swap))

with pd.ExcelWriter(OUT / 'tables' / 'v1_parts.xlsx', engine='xlsxwriter') as xw:
    for k_, v_ in OUTX.items(): v_.to_excel(xw, sheet_name=k_[:31], index=False)
for k_ in ['A_element_prevalence', 'BC_tests', 'D_element_by_element', 'EF_parts_low_end']: display(OUTX[k_].round(3))
print('saved:', OUT / 'tables' / 'v1_parts.xlsx')

### 7. Parts and whole: K-pop vs indie, choruses by ear, indie with performers

**فارسی:** آیا جزءها به ویدئوی خودشان تعلق دارند؟ کی‌پاپ در برابر ایندی با کورس‌های گوشی؛ ایندی با اجراکننده.

*Thesis: 3-7; 4-3-6, 4-8-5*

In [ ]:
# 7. Do parts belong to their own video? K-pop vs Korean indie (Adorno: interchangeable parts vs parts bound to the whole)
#   Uses only choruses found by ear: 16 K-pop choruses (eye-tracking shortlist, listed below) and the indie onsets in tables/indie_chorus_check.xlsx.
#   Part (3) repeats the visual coherence with indie videos that show performers (tables/indie_composition_filled.xlsx). -> tables/parts_kpop_indie.xlsx
# Thesis: 3-7; 4-3-6, 4-8-5
import re, unicodedata
from scipy.stats import mannwhitneyu
rng = np.random.default_rng(25); R = {}
norm = lambda s: re.sub(r'[\s_\-]+', '', unicodedata.normalize('NFC', str(s))).lower()

def segs(rec):                                         # 8-second segments of one video, unit length, with their start times
    W = np.asarray(rec['W'], float); W = W / np.linalg.norm(W, axis=1, keepdims=True)
    st = np.asarray(rec.get('starts', np.arange(len(W)) * 4.0), float); return W, st
KP = {x: segs(K[x]) for x in names}; ID = {x: segs(Ii[x]) for x in Ii}
art_k = T.set_index('file_name').artist.reindex(names).to_dict()
art = {**{x: art_k[x] for x in names}, **{x: x.split(' - ')[0].split('_')[0].strip() for x in ID}}
ALL = {**{('kpop', x): v for x, v in KP.items()}, **{('indie', x): v for x, v in ID.items()}}

# (1) WHOLE VIDEO: own-video belonging = similarity of a segment to the rest of its OWN video (not adjacent, |gap| > 8 s)
#     minus its similarity to segments of all OTHER acts' videos (both corpora pooled, same reference for everybody).
pool = np.vstack([v[0] for v in ALL.values()]); pool_key = np.concatenate([[k] * len(v[0]) for k, v in ALL.items()], axis=0)
pool_art = np.array([art[k[1]] for k in pool_key])
rows = []
for (c, x), (W, st) in ALL.items():
    Sown = W @ W.T; far = np.abs(st[:, None] - st[None, :]) > 8; np.fill_diagonal(far, False)
    if far.sum() == 0: continue
    own = Sown[far].mean(); other = (W @ pool[pool_art != art[x]].T).mean()
    rows.append(dict(corpus=c, file_name=x, act=art[x], own_rest=own, other_acts=other, belonging=own - other, n_segments=len(W)))
B = pd.DataFrame(rows); R['video_belonging'] = B

def act_boot(df, col):                                  # act-level bootstrap of the K-pop minus indie difference
    gk, gi = [g for _, g in df[df.corpus == 'kpop'].groupby('act')], [g for _, g in df[df.corpus == 'indie'].groupby('act')]
    d = []
    for _ in range(2000):
        a = pd.concat([gk[i] for i in rng.integers(0, len(gk), len(gk))]); b = pd.concat([gi[i] for i in rng.integers(0, len(gi), len(gi))])
        d.append(a[col].mean() - b[col].mean())
    return np.percentile(d, [2.5, 97.5])
def cliff(a, b): a, b = np.asarray(a), np.asarray(b); return np.sign(a[:, None] - b[None, :]).mean()
def summary(df, col, label):
    k, i = df[df.corpus == 'kpop'][col], df[df.corpus == 'indie'][col]; lo, hi = act_boot(df, col)
    return dict(test=label, kpop_mean=k.mean(), indie_mean=i.mean(), kpop_minus_indie=k.mean() - i.mean(), ci_low_act_boot=lo, ci_high_act_boot=hi,
                cliff_delta=cliff(k, i), p_mannwhitney=mannwhitneyu(k, i)[1], n_kpop=len(k), n_indie=len(i))
S1 = [summary(B, 'belonging', 'whole video: own-rest minus other-acts similarity (higher = parts belong to their own video)'),
      summary(B, 'own_rest', 'whole video: similarity to own rest only (internal unity; also raised by one set/outfit)')]
#     control for the number of looks: K-pop videos change set/outfit more; residualize belonging on the video's own internal spread
B['spread'] = 1 - B['own_rest']
b_ = np.polyfit(B.spread, B.belonging, 1); B['belonging_adj'] = B.belonging - np.polyval(b_, B.spread)
S1.append(summary(B, 'belonging_adj', 'whole video: belonging adjusted for internal visual variety (sets/outfits)'))

# (2) CHORUS TO CHORUS (by ear only): is a chorus closer to the rest of its own video than to the OTHER choruses?
KCH = {'aespa 에스파  Supernova  MV_720p.mp4': 41, 'BABYMONSTER -  춤 (CHOOM)  M_V_720p.mp4': 67, 'NMIXX  O.O  M_V_720p.mp4': 63,
       'KISS OF LIFE (키스오브라이프)  Sticky  Official Music Video_720p.mp4': 53, 'NewJeans (뉴진스)  Super Shy  Official MV_720p.mp4': 66,
       'BABYMONSTER -  I LIKE IT  M_V_720p.mp4': 60, 'Stray Kids  神메뉴(God s Menu)  M_V_720p.mp4': 53, 'ZEROBASEONE (제로베이스원)  GOOD SO BAD  MV_720p.mp4': 58,
       'SHINee 샤이니  HARD  MV_720p.mp4': 53, 'V ‘FRI(END)S’ Official MV_720p.mp4': 93, 'BTS (방탄소년단)  Take Two  MV_720p.mp4': 67,
       'SEVENTEEN (세븐틴)  손오공  Official MV_720p.mp4': 70, 'TWICE  SET ME FREE  M_V_720p.mp4': 45,
       'LE SSERAFIM (르세라핌)  ANTIFRAGILE  OFFICIAL M_V_720p.mp4': 64, 'RIIZE 라이즈  Love 119  MV_720p.mp4': 91,
       'BOYNEXTDOOR (보이넥스트도어)  돌아버리겠다  Official MV_720p.mp4': 44}
def secs(v):
    if pd.isna(v) or str(v).strip() == '': return np.nan
    s_ = str(v).strip().replace('٫', '.'); return 60 * float(s_.split(':')[0]) + float(s_.split(':')[1]) if ':' in s_ else float(s_)
ck = pd.read_excel(OUT / 'tables' / 'indie_chorus_check.xlsx')
col = 'chorus_start' if 'chorus_start' in ck else 'true_chorus_s'
ICH = {r.file_name: secs(getattr(r, col)) for r in ck.itertuples() if not pd.isna(secs(getattr(r, col)))}
def match(d, keys):
    m = {norm(k): k for k in keys}; out = {}
    for k, v in d.items():
        if norm(k) in m: out[m[norm(k)]] = v
        else: print('no match, skipped:', k)
    return out
KCH, ICH = match(KCH, KP), match(ICH, ID); print('choruses used: K-pop', len(KCH), '| indie', len(ICH))
CH = {}
for c, D, src in [('kpop', KCH, KP), ('indie', ICH, ID)]:
    for x, t0 in D.items():
        W, st = src[x]; j = int(np.argmin(np.abs(st - t0)))     # the 8-second segment that starts closest to the chorus onset
        CH[(c, x)] = (W[j], W[np.abs(st - st[j]) > 8])          # chorus segment, rest of own video (not overlapping)
rows = []
for (c, x), (ch, rest) in CH.items():
    others = [v[0] for (c2, x2), v in CH.items() if art[x2] != art[x]]
    rows.append(dict(corpus=c, file_name=x, act=art[x], chorus_to_own_rest=(rest @ ch).mean(),
                     chorus_to_other_choruses=np.mean([o @ ch for o in others]),
                     chorus_to_other_choruses_same_corpus=np.mean([v[0] @ ch for (c2, x2), v in CH.items() if c2 == c and art[x2] != art[x]])))
C = pd.DataFrame(rows); C['chorus_belonging'] = C.chorus_to_own_rest - C.chorus_to_other_choruses; R['chorus_belonging'] = C
S1 += [summary(C, 'chorus_belonging', 'chorus: own-rest minus other choruses (by ear; higher = chorus bound to its own video)'),
       summary(C, 'chorus_to_other_choruses_same_corpus', 'chorus: similarity to other choruses of the same corpus (higher = choruses interchangeable)')]
#     swap test: does the chorus's nearest neighbour among ALL segments of the other videos + its own rest lie in its own video?
sw = []
for (c, x), (ch, rest) in CH.items():
    best_own = (rest @ ch).max() if len(rest) else -1
    best_other = max(v[0] @ ch for (c2, x2), v in CH.items() if art[x2] != art[x])
    sw.append(dict(corpus=c, file_name=x, nearest_is_other_chorus=best_other > best_own))
SW = pd.DataFrame(sw); R['chorus_swap'] = SW
S1.append(dict(test='share of choruses whose closest match is ANOTHER act\'s chorus (vs own video)',
               kpop_mean=SW[SW.corpus == 'kpop'].nearest_is_other_chorus.mean(), indie_mean=SW[SW.corpus == 'indie'].nearest_is_other_chorus.mean(),
               n_kpop=int((SW.corpus == 'kpop').sum()), n_indie=int((SW.corpus == 'indie').sum())))
# (3) Is the K-pop/indie gap only "videos with people vs videos without"? Repeat the visual coherence with indie videos that show performers.
#     Needs tables/indie_composition_filled.xlsx (columns file_name, performer, animation).
try:
    comp = pd.read_excel(OUT / 'tables' / 'indie_composition_filled.xlsx')
    with_perf = {norm(r.file_name) for r in comp.itertuples() if str(r.performer).strip().lower() not in ('no performer', '0', 'nan') and str(r.animation).strip().lower() not in ('yes', '1')}
    vec = lambda rec: (lambda v: v / np.linalg.norm(v))(np.asarray(rec['v'], float))
    Vk_ = {x: vec(K[x]) for x in names}; Vi_all = {x: vec(Ii[x]) for x in Ii}; Vi_p = {x: v for x, v in Vi_all.items() if norm(x) in with_perf}
    def coh_one_per_act(D, n, draws=2000):
        acts_ = pd.Series({x: art[x] for x in D}); out = []
        for _ in range(draws):
            pick = [rng.choice(g.index.to_numpy()) for _, g in acts_.groupby(acts_)]
            pick = rng.choice(pick, min(n, len(pick)), replace=False); M = np.stack([D[x] for x in pick]); S_ = M @ M.T
            out.append(S_[np.triu_indices(len(M), 1)].mean())
        return np.array(out)
    n_p = len({art[x] for x in Vi_p})
    ck_, ci_p, ci_a = coh_one_per_act(Vk_, n_p), coh_one_per_act(Vi_p, n_p), coh_one_per_act(Vi_all, n_p)
    S1 += [dict(test=f'visual coherence, K-pop vs indie WITH performers ({len(Vi_p)} videos, {n_p} acts; one per act, same n)',
                kpop_mean=np.median(ck_), indie_mean=np.median(ci_p), kpop_minus_indie=np.median(ck_ - ci_p),
                ci_low_act_boot=np.percentile(ck_ - ci_p, 2.5), ci_high_act_boot=np.percentile(ck_ - ci_p, 97.5), n_kpop=n_p, n_indie=n_p),
           dict(test=f'visual coherence, K-pop vs ALL indie (same n = {n_p}, for comparison)',
                kpop_mean=np.median(ck_), indie_mean=np.median(ci_a), kpop_minus_indie=np.median(ck_ - ci_a),
                ci_low_act_boot=np.percentile(ck_ - ci_a, 2.5), ci_high_act_boot=np.percentile(ck_ - ci_a, 97.5), n_kpop=n_p, n_indie=n_p)]
    if 'chorus_belonging' in C:
        C['indie_has_performer'] = [norm(x) in with_perf if c == 'indie' else np.nan for c, x in zip(C.corpus, C.file_name)]
except FileNotFoundError:
    print('indie_composition_filled.xlsx not in tables: performer check skipped')
R['summary'] = pd.DataFrame(S1)
with pd.ExcelWriter(OUT / 'tables' / 'parts_kpop_indie.xlsx', engine='xlsxwriter') as xw:
    for k_, v_ in R.items(): v_.to_excel(xw, sheet_name=k_[:31], index=False)
display(R['summary'].round(3)); print('saved:', OUT / 'tables' / 'parts_kpop_indie.xlsx')

### 8. Expert panel: I-CVI, modified kappa, S-CVI/Ave

**فارسی:** پنل خبرگان: شاخص اعتبار محتوایی هر کد (I-CVI)، کاپای تعدیل‌شده و S-CVI/Ave از برگه‌های بازگشتی.

*Thesis: 3-6-3; Table 3-5b*

In [ ]:
# 8. Expert-panel content validity (Lynn, 1986; Polit & Beck, 2006; Polit et al., 2007) -> tables/cvi_results.xlsx
#   I-CVI = share of experts rating relevance 3 or 4; modified kappa k* = (I-CVI - Pc) / (1 - Pc), Pc = C(N, A) * 0.5**N;
#   S-CVI/Ave = mean I-CVI over codes; clarity is reported as the share of experts rating it 3 or 4.
# Thesis: 3-6-3; Table 3-5b
from math import comb
files = sorted(EXPERT_DIR.glob('expert_review_*.xlsx'))
ID_COL, REL_COL, CLA_COL = 'شناسه', 'ارتباط (۱ تا ۴)', 'وضوح (۱ تا ۴)'      # column names of the 'داوری' sheet
R_, C_ = {}, {}
for k, f in enumerate(files, 1):
    d = pd.read_excel(f, sheet_name='داوری')
    d = d[d[ID_COL].astype(str).str.match(r'^(el_|share_|typicality)')]          # code rows only (no notes / name rows)
    R_[k] = pd.to_numeric(d.set_index(ID_COL)[REL_COL], errors='coerce'); C_[k] = pd.to_numeric(d.set_index(ID_COL)[CLA_COL], errors='coerce')
REL_R, CLA_R = pd.DataFrame(R_), pd.DataFrame(C_)
print('experts:', len(files), '| codes rated:', len(REL_R))
def kappa_star(icvi, n):
    a = int(round(icvi * n)); pc = comb(n, a) * 0.5 ** n; return (icvi - pc) / (1 - pc)
CVI = pd.DataFrame({'relevance': REL_R.apply(lambda r: '/'.join(str(int(v)) for v in r.dropna()), axis=1),
                    'I_CVI': (REL_R >= 3).sum(axis=1) / REL_R.notna().sum(axis=1),
                    'clarity': CLA_R.apply(lambda r: '/'.join(str(int(v)) for v in r.dropna()), axis=1),
                    'clarity_share_3_4': (CLA_R >= 3).sum(axis=1) / CLA_R.notna().sum(axis=1)})
CVI['kappa_star'] = [kappa_star(i, n) for i, n in zip(CVI.I_CVI, REL_R.notna().sum(axis=1))]
CVI = CVI.rename_axis(ID_COL).reset_index()
SCVI = pd.DataFrame([dict(quantity='S-CVI/Ave (mean I-CVI over codes)', value=CVI.I_CVI.mean()),
                     dict(quantity='codes with I-CVI = 1.00', value=int((CVI.I_CVI == 1).sum())),
                     dict(quantity='number of experts', value=len(files))])
with pd.ExcelWriter(OUT / 'tables' / 'cvi_results.xlsx', engine='xlsxwriter') as xw:
    CVI.to_excel(xw, sheet_name='Sheet1', index=False); SCVI.to_excel(xw, sheet_name='S_CVI', index=False)
display(CVI.round(3)); display(SCVI.round(3))